# Pretrained Embedding Fine-Tuning

이 노트북은 MPOSE2021로 사전학습한 scratch 계열 TCN/GCN encoder를 로컬 댄스 reference 데이터로 fine-tuning하고, 서비스의 `embedding` 옵션에서 사용할 `.tflite` 모델까지 확인하는 end-to-end 노트북이다.

`scratch_tflite_training.ipynb`와 같은 큰 구조를 따른다.

1. 환경과 경로 확인
2. fine-tuning에 사용할 댄스 skeleton 데이터 확인
3. 사전학습 encoder 목록 확인
4. 여러 fine-tuning config를 순차 학습
5. `.keras`, `.tflite`, metadata 결과 확인
6. 학습된 encoder의 유사도를 애니메이션과 cosine matrix로 확인
7. 서비스 runtime smoke test와 실행 명령 확인

입력:

```text
data/models/pretrain/mpose2021/*_encoder.keras
```

출력:

```text
data/models/embedding/{name}_encoder.keras
data/models/embedding/{name}.tflite
data/models/embedding/{name}_meta.json
```


## 1. 환경과 프로젝트 루트 확인

노트북이 어느 위치에서 열려도 `pjt_main` 루트를 찾아서 이후 셀들이 같은 경로 기준으로 동작하도록 한다.


In [1]:
from pathlib import Path
import json
import subprocess
import sys
import time


def find_project_root(start=None):
    start = Path(start or Path.cwd()).resolve()
    for path in [start, *start.parents]:
        if (path / 'scripts' / 'finetune_embedding.py').exists():
            return path
    raise RuntimeError('Could not find pjt_main project root')


PROJECT_ROOT = find_project_root()
for import_path in (PROJECT_ROOT, PROJECT_ROOT / 'src'):
    import_path = str(import_path)
    if import_path not in sys.path:
        sys.path.insert(0, import_path)

print('PROJECT_ROOT =', PROJECT_ROOT)
print('PYTHON =', sys.executable)


PROJECT_ROOT = /workspace/users/yijin/boot_env/pjt_main
PYTHON = /workspace/users/yijin/boot_env/.venv/bin/python


## 2. 경로와 사전학습 Encoder 확인

fine-tuning의 시작점은 `data/models/pretrain/mpose2021/*_encoder.keras`이다. 각 encoder는 MPOSE2021 skeleton action 데이터로 먼저 학습된 뒤, 이 노트북에서 로컬 댄스 reference 데이터에 맞게 추가 학습된다.


In [2]:
# Paths
PRETRAIN_DIR = PROJECT_ROOT / 'data' / 'models' / 'pretrain' / 'mpose2021'
DANCE_DATA_DIR = PROJECT_ROOT / 'data' / 'reference_dances'
OUTPUT_DIR = PROJECT_ROOT / 'data' / 'models' / 'embedding'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print('PRETRAIN_DIR =', PRETRAIN_DIR)
print('DANCE_DATA_DIR =', DANCE_DATA_DIR)
print('OUTPUT_DIR =', OUTPUT_DIR)

print('\nAvailable pretrained encoders:')
for path in sorted(PRETRAIN_DIR.glob('*_encoder.keras')):
    print(' -', path.name)

PRETRAIN_DIR = /workspace/users/yijin/boot_env/pjt_main/data/models/pretrain/mpose2021
DANCE_DATA_DIR = /workspace/users/yijin/boot_env/pjt_main/data/reference_dances
OUTPUT_DIR = /workspace/users/yijin/boot_env/pjt_main/data/models/embedding

Available pretrained encoders:
 - mpose2021_gcn_e32_triplet_encoder.keras
 - mpose2021_gcn_e64_triplet_encoder.keras
 - mpose2021_tcn_e32_triplet_encoder.keras
 - mpose2021_tcn_e64_triplet_encoder.keras


## 3. Dance Reference 데이터 확인

fine-tuning 원본은 `data/reference_dances/*/reference.npy`이다. shape는 보통 `(frames, 33, 4)`이고 마지막 차원은 `x, y, z, visibility`이다.

모델 입력으로는 전체 33개 landmark가 아니라 `DANCE_JOINTS` 12개 관절을 선택하고, scratch 모델과 같은 전처리를 적용한다.


In [3]:
# Inspect local dance reference data.
import numpy as np

rows = []
total_frames = 0
for ref_path in sorted(DANCE_DATA_DIR.glob('*/reference.npy')):
    arr = np.load(ref_path, allow_pickle=True)
    total_frames += int(arr.shape[0])
    rows.append((ref_path.parent.name, tuple(arr.shape), arr.shape[0] / 30.0))

print('name\tshape\tduration@30fps')
for name, shape, seconds in rows:
    print(f'{name}\t{shape}\t{seconds:.1f}s')
print(f'TOTAL\t{total_frames} frames\t{total_frames / 30.0:.1f}s')

name	shape	duration@30fps
beginner_wave	(1800, 33, 4)	60.0s
cheerup_dance	(724, 33, 4)	24.1s
freestyle_free	(3600, 33, 4)	120.0s
hiphop_move	(792, 33, 4)	26.4s
kpop_basic	(2700, 33, 4)	90.0s
TOTAL	9616 frames	320.5s


## 4. Fine-Tuning 설정

`FINETUNE_CONFIGS`에 여러 모델을 넣으면 scratch training 노트북처럼 순차적으로 학습한다.

- `pretrained_name`: 로드할 사전학습 encoder 이름
- `name`: fine-tuning 후 저장할 embedding 모델 이름
- `loss_type`: `triplet` 또는 `bce`

현재 없는 pretrained encoder는 `SKIP_MISSING_PRETRAINED=True`일 때 자동으로 건너뛴다.


In [4]:
# Fine-tuning settings.
# Defaults are smoke-run friendly. Increase epochs/steps for real training.
EPOCHS = 30
STEPS_PER_EPOCH = 60
VALIDATION_STEPS = 10
BATCH_SIZE = 32
PATIENCE = 5
LEARNING_RATE = 5e-4
POSITIVE_JITTER = 6
NEGATIVE_GAP = 600
NOISE_STD = 0.015

# Set ONLY to a list of fine-tuned names to train a subset.
ONLY = []
SKIP_MISSING_PRETRAINED = True

FINETUNE_CONFIGS = [
    {
        'pretrained_name': 'mpose2021_tcn_e32_triplet',
        'name': 'embedding_tcn_e32_triplet',
        'loss_type': 'triplet',
    },
    {
        'pretrained_name': 'mpose2021_tcn_e64_triplet',
        'name': 'embedding_tcn_e64_triplet',
        'loss_type': 'triplet',
    },
    {
        'pretrained_name': 'mpose2021_gcn_e32_triplet',
        'name': 'embedding_gcn_e32_triplet',
        'loss_type': 'triplet',
    },
    {
        'pretrained_name': 'mpose2021_gcn_e64_triplet',
        'name': 'embedding_gcn_e64_triplet',
        'loss_type': 'triplet',
    },
    # Optional BCE fine-tuning baselines if matching pretrained BCE encoders exist.
    # {'pretrained_name': 'mpose2021_gcn_e64_bce', 'name': 'dance_embedding_gcn_e64_bce', 'loss_type': 'bce'},
    # {'pretrained_name': 'mpose2021_tcn_e64_bce', 'name': 'dance_embedding_tcn_e64_bce', 'loss_type': 'bce'},
]

SELECTED_CONFIGS = [cfg for cfg in FINETUNE_CONFIGS if not ONLY or cfg['name'] in ONLY]
assert SELECTED_CONFIGS, 'No configs selected'

print('selected configs:')
for cfg in SELECTED_CONFIGS:
    pretrained_path = PRETRAIN_DIR / f"{cfg['pretrained_name']}_encoder.keras"
    print(' -', cfg['name'], '<-', pretrained_path.name, 'exists=', pretrained_path.exists())

selected configs:
 - embedding_tcn_e32_triplet <- mpose2021_tcn_e32_triplet_encoder.keras exists= True
 - embedding_tcn_e64_triplet <- mpose2021_tcn_e64_triplet_encoder.keras exists= True
 - embedding_gcn_e32_triplet <- mpose2021_gcn_e32_triplet_encoder.keras exists= True
 - embedding_gcn_e64_triplet <- mpose2021_gcn_e64_triplet_encoder.keras exists= True


## 5. Fine-Tuning 실행

각 config는 `scripts/finetune_embedding.py`를 호출한다. 이 스크립트는 pretrained `.keras` encoder를 로드하고, 로컬 댄스 reference에서 positive/negative 또는 triplet window를 샘플링해 추가 학습한다.

저장되는 것은 서비스용 shared encoder 하나다. Siamese/triplet 전체 모델이 아니라 encoder만 `.keras`와 `.tflite`로 export한다.


In [6]:
# Fine-tune every selected variant.
# Outputs:
#   data/models/embedding/{name}_encoder.keras
#   data/models/embedding/{name}.tflite
#   data/models/embedding/{name}_meta.json

FINETUNE_RESULTS = []

for index, cfg in enumerate(SELECTED_CONFIGS, start=1):
    pretrained_path = PRETRAIN_DIR / f"{cfg['pretrained_name']}_encoder.keras"
    if not pretrained_path.exists():
        msg = f"Missing pretrained encoder: {pretrained_path}"
        if SKIP_MISSING_PRETRAINED:
            print('[SKIP]', msg)
            continue
        raise FileNotFoundError(msg)

    print('=' * 100)
    print(f"[{index}/{len(SELECTED_CONFIGS)}] Fine-tuning {cfg['name']}")
    print('=' * 100)

    cmd = [
        sys.executable, str(PROJECT_ROOT / 'scripts' / 'finetune_embedding.py'),
        '--data-dir', str(DANCE_DATA_DIR),
        '--pretrained-encoder', str(pretrained_path),
        '--model-name', cfg['name'],
        '--output-dir', str(OUTPUT_DIR),
        '--loss-type', cfg.get('loss_type', 'triplet'),
        '--epochs', str(cfg.get('epochs', EPOCHS)),
        '--steps-per-epoch', str(cfg.get('steps_per_epoch', STEPS_PER_EPOCH)),
        '--validation-steps', str(cfg.get('validation_steps', VALIDATION_STEPS)),
        '--batch-size', str(cfg.get('batch_size', BATCH_SIZE)),
        '--patience', str(cfg.get('patience', PATIENCE)),
        '--learning-rate', str(cfg.get('learning_rate', LEARNING_RATE)),
        '--positive-jitter', str(cfg.get('positive_jitter', POSITIVE_JITTER)),
        '--negative-gap', str(cfg.get('negative_gap', NEGATIVE_GAP)),
        '--noise-std', str(cfg.get('noise_std', NOISE_STD)),
    ]
    if 'triplet_margin' in cfg:
        cmd += ['--triplet-margin', str(cfg['triplet_margin'])]
    if cfg.get('no_quantize', False):
        cmd += ['--no-quantize']

    started = time.time()
    print(' '.join(cmd))
    subprocess.check_call(cmd, cwd=PROJECT_ROOT)
    elapsed = time.time() - started

    meta_path = OUTPUT_DIR / f"{cfg['name']}_meta.json"
    meta = json.loads(meta_path.read_text(encoding='utf-8')) if meta_path.exists() else {}
    FINETUNE_RESULTS.append({
        'name': cfg['name'],
        'pretrained_name': cfg['pretrained_name'],
        'loss_type': cfg.get('loss_type', 'triplet'),
        'elapsed_sec': round(elapsed, 1),
        'training_summary': meta.get('training_summary', {}),
        'smoke_metrics': meta.get('smoke_metrics', {}),
        'keras': str(OUTPUT_DIR / f"{cfg['name']}_encoder.keras"),
        'tflite': str(OUTPUT_DIR / f"{cfg['name']}.tflite"),
        'meta': str(meta_path),
        'service_command': f"python src/main.py -s embedding --embedding-model-name {cfg['name']}",
    })

print()
print('All selected fine-tuning runs finished.')
for result in FINETUNE_RESULTS:
    print(result)

[1/4] Fine-tuning embedding_tcn_e32_triplet
/workspace/users/yijin/boot_env/.venv/bin/python /workspace/users/yijin/boot_env/pjt_main/scripts/finetune_embedding.py --data-dir /workspace/users/yijin/boot_env/pjt_main/data/reference_dances --pretrained-encoder /workspace/users/yijin/boot_env/pjt_main/data/models/pretrain/mpose2021/mpose2021_tcn_e32_triplet_encoder.keras --model-name embedding_tcn_e32_triplet --output-dir /workspace/users/yijin/boot_env/pjt_main/data/models/embedding --loss-type triplet --epochs 30 --steps-per-epoch 60 --validation-steps 10 --batch-size 32 --patience 5 --learning-rate 0.0005 --positive-jitter 6 --negative-gap 600 --noise-std 0.015
[DATA] loaded 5 dance reference sequences
  - beginner_wave: 1800 frames (33, 4)
  - hiphop_move: 792 frames (33, 4)
  - kpop_basic: 2700 frames (33, 4)
  - cheerup_dance: 724 frames (33, 4)
  - freestyle_free: 3600 frames (33, 4)


2026-04-16 10:29:20.997696: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:479] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
2026-04-16 10:29:21.011638: E external/local_xla/xla/stream_executor/cuda/cuda_dnn.cc:10575] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
2026-04-16 10:29:21.011662: E external/local_xla/xla/stream_executor/cuda/cuda_blas.cc:1442] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
2026-04-16 10:29:21.021368: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
2026-04-16 10:29:21.594352: W tensorflow/compiler/tf

[MODEL] encoder config: {'sequence_length': 30, 'num_joints': 12, 'feature_dims': 2, 'embedding_dim': 32}
Model: "scratch_triplet_similarity"
┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ anchor_window       │ (None, 30, 12, 2) │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼��───────────┼───────────────────┤
│ positive_window     │ (None, 30, 12, 2) │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ negative_window     │ (None, 30, 12, 2) │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────

2026-04-16 10:30:32.164314: I external/local_xla/xla/stream_executor/cuda/cuda_executor.cc:998] successful NUMA node read from SysFS had negative value (-1), but there must be at least one NUMA node, so returning NUMA node zero. See more at https://github.com/torvalds/linux/blob/v6.0/Documentation/ABI/testing/sysfs-bus-pci#L344-L355
2026-04-16 10:30:32.168690: I tensorflow/core/grappler/devices.cc:66] Number of eligible GPUs (core count >= 8, compute capability >= 0.0): 1
2026-04-16 10:30:32.168761: I tensorflow/core/grappler/clusters/single_machine.cc:361] Starting new session
2026-04-16 10:30:32.168986: I external/local_xla/xla/stream_executor/cuda/cuda_executor.cc:998] successful NUMA node read from SysFS had negative value (-1), but there must be at least one NUMA node, so returning NUMA node zero. See more at https://github.com/torvalds/linux/blob/v6.0/Documentation/ABI/testing/sysfs-bus-pci#L344-L355
2026-04-16 10:30:32.171509: W tensorflow/core/common_runtime/gpu/gpu_device.cc:2

[SAVE] fine-tuned TFLite encoder: /workspace/users/yijin/boot_env/pjt_main/data/models/embedding/embedding_tcn_e32_triplet.tflite (97.1 KiB)
[VERIFY] input=[1, 30, 12, 2] output=[1, 32] embedding_norm=4.5998
[BEST] {'monitor': 'val_loss', 'restore_best_weights': True, 'epochs_ran': 10, 'best_epoch': 5, 'best_val_loss': 0.0031853322871029377, 'best_epoch_loss': 0.008410806767642498}
[METRIC] {'same_cosine_mean': 0.9440389373339713, 'different_cosine_mean': 0.16111386167176533, 'margin_mean': 0.7829250756622059}
[SAVE] metadata: /workspace/users/yijin/boot_env/pjt_main/data/models/embedding/embedding_tcn_e32_triplet_meta.json

Run with:
  python src/main.py -s embedding --embedding-model-name embedding_tcn_e32_triplet
[2/4] Fine-tuning embedding_tcn_e64_triplet
/workspace/users/yijin/boot_env/.venv/bin/python /workspace/users/yijin/boot_env/pjt_main/scripts/finetune_embedding.py --data-dir /workspace/users/yijin/boot_env/pjt_main/data/reference_dances --pretrained-encoder /workspace/user

...

[SAVE] fine-tuned TFLite encoder: /workspace/users/yijin/boot_env/pjt_main/data/models/embedding/embedding_gcn_e32_triplet.tflite (466.4 KiB)
[VERIFY] input=[1, 30, 12, 2] output=[1, 32] embedding_norm=29.1956
[BEST] {'monitor': 'val_loss', 'restore_best_weights': True, 'epochs_ran': 26, 'best_epoch': 21, 'best_val_loss': 0.0028650914318859577, 'best_epoch_loss': 0.007876457646489143}
[METRIC] {'same_cosine_mean': 0.9470612006261945, 'different_cosine_mean': 0.008684805157827213, 'margin_mean': 0.9383763954683673}
[SAVE] metadata: /workspace/users/yijin/boot_env/pjt_main/data/models/embedding/embedding_gcn_e32_triplet_meta.json

Run with:
  python src/main.py -s embedding --embedding-model-name embedding_gcn_e32_triplet
[4/4] Fine-tuning embedding_gcn_e64_triplet
/workspace/users/yijin/boot_env/.venv/bin/python /workspace/users/yijin/boot_env/pjt_main/scripts/finetune_embedding.py --data-dir /workspace/users/yijin/boot_env/pjt_main/data/reference_dances --pretrained-encoder /workspace/

2026-04-16 10:36:35.622976: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:479] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
2026-04-16 10:36:35.637546: E external/local_xla/xla/stream_executor/cuda/cuda_dnn.cc:10575] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
2026-04-16 10:36:35.637568: E external/local_xla/xla/stream_executor/cuda/cuda_blas.cc:1442] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
2026-04-16 10:36:35.647361: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
2026-04-16 10:36:36.215967: W tensorflow/compiler/tf

[MODEL] encoder config: {'sequence_length': 30, 'num_joints': 12, 'feature_dims': 2, 'embedding_dim': 64}
Model: "scratch_triplet_similarity"
┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ anchor_window       │ (None, 30, 12, 2) │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼��───────────┼───────────────────┤
│ positive_window     │ (None, 30, 12, 2) │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ negative_window     │ (None, 30, 12, 2) │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────

2026-04-16 10:39:52.344641: I external/local_xla/xla/stream_executor/cuda/cuda_executor.cc:998] successful NUMA node read from SysFS had negative value (-1), but there must be at least one NUMA node, so returning NUMA node zero. See more at https://github.com/torvalds/linux/blob/v6.0/Documentation/ABI/testing/sysfs-bus-pci#L344-L355
2026-04-16 10:39:52.348764: I tensorflow/core/grappler/devices.cc:66] Number of eligible GPUs (core count >= 8, compute capability >= 0.0): 1
2026-04-16 10:39:52.348844: I tensorflow/core/grappler/clusters/single_machine.cc:361] Starting new session
2026-04-16 10:39:52.349106: I external/local_xla/xla/stream_executor/cuda/cuda_executor.cc:998] successful NUMA node read from SysFS had negative value (-1), but there must be at least one NUMA node, so returning NUMA node zero. See more at https://github.com/torvalds/linux/blob/v6.0/Documentation/ABI/testing/sysfs-bus-pci#L344-L355
2026-04-16 10:39:52.351639: W tensorflow/core/common_runtime/gpu/gpu_device.cc:2

[SAVE] fine-tuned TFLite encoder: /workspace/users/yijin/boot_env/pjt_main/data/models/embedding/embedding_gcn_e64_triplet.tflite (468.5 KiB)
[VERIFY] input=[1, 30, 12, 2] output=[1, 64] embedding_norm=30.7494
[BEST] {'monitor': 'val_loss', 'restore_best_weights': True, 'epochs_ran': 21, 'best_epoch': 16, 'best_val_loss': 0.0020462842658162117, 'best_epoch_loss': 0.005147154908627272}
[METRIC] {'same_cosine_mean': 0.9510154817253351, 'different_cosine_mean': 0.05021176023001317, 'margin_mean': 0.900803721495322}
[SAVE] metadata: /workspace/users/yijin/boot_env/pjt_main/data/models/embedding/embedding_gcn_e64_triplet_meta.json

Run with:
  python src/main.py -s embedding --embedding-model-name embedding_gcn_e64_triplet

All selected fine-tuning runs finished.
{'name': 'embedding_tcn_e32_triplet', 'pretrained_name': 'mpose2021_tcn_e32_triplet', 'loss_type': 'triplet', 'elapsed_sec': 77.7, 'training_summary': {'monitor': 'val_loss', 'restore_best_weights': True, 'epochs_ran': 10, 'best_ep

## 5.1 결과 파일과 Metadata 확인

각 모델의 `training_summary`에는 best epoch, best validation loss, 실제로 몇 epoch을 돌았는지가 들어간다. `smoke_metrics`는 같은 춤의 가까운 window와 다른 춤 window의 cosine 차이가 어느 정도인지 빠르게 보는 참고값이다.


In [7]:
# Check generated artifacts and compact result table.
if 'FINETUNE_RESULTS' not in globals():
    FINETUNE_RESULTS = []
    for cfg in SELECTED_CONFIGS:
        meta_path = OUTPUT_DIR / f"{cfg['name']}_meta.json"
        if not meta_path.exists():
            continue
        meta = json.loads(meta_path.read_text(encoding='utf-8'))
        FINETUNE_RESULTS.append({
            'name': cfg['name'],
            'pretrained_name': cfg['pretrained_name'],
            'loss_type': cfg.get('loss_type', 'triplet'),
            'training_summary': meta.get('training_summary', {}),
            'smoke_metrics': meta.get('smoke_metrics', {}),
            'keras': str(OUTPUT_DIR / f"{cfg['name']}_encoder.keras"),
            'tflite': str(OUTPUT_DIR / f"{cfg['name']}.tflite"),
            'meta': str(meta_path),
            'service_command': f"python src/main.py -s embedding --embedding-model-name {cfg['name']}",
        })

for result in FINETUNE_RESULTS:
    print('-' * 100)
    print(result['name'], '<-', result['pretrained_name'])
    print('  best:', result.get('training_summary', {}))
    print('  smoke:', result.get('smoke_metrics', {}))
    for key in ('keras', 'tflite', 'meta'):
        path = Path(result[key])
        size = f'{path.stat().st_size / 1024:.1f} KiB' if path.exists() else 'missing'
        print(f'  {key}: {path} ({size})')
    print('  run:', result['service_command'])

----------------------------------------------------------------------------------------------------
embedding_tcn_e32_triplet <- mpose2021_tcn_e32_triplet
  best: {'monitor': 'val_loss', 'restore_best_weights': True, 'epochs_ran': 10, 'best_epoch': 5, 'best_val_loss': 0.0031853322871029377, 'best_epoch_loss': 0.008410806767642498}
  smoke: {'same_cosine_mean': 0.9440389373339713, 'different_cosine_mean': 0.16111386167176533, 'margin_mean': 0.7829250756622059}
  keras: /workspace/users/yijin/boot_env/pjt_main/data/models/embedding/embedding_tcn_e32_triplet_encoder.keras (428.7 KiB)
  tflite: /workspace/users/yijin/boot_env/pjt_main/data/models/embedding/embedding_tcn_e32_triplet.tflite (97.1 KiB)
  meta: /workspace/users/yijin/boot_env/pjt_main/data/models/embedding/embedding_tcn_e32_triplet_meta.json (4.4 KiB)
  run: python src/main.py -s embedding --embedding-model-name embedding_tcn_e32_triplet
-----------------------------------------------------------------------------------------

## 6. 유사도와 애니메이션 확인

아래 셀은 fine-tuned TFLite encoder가 실제 댄스 window들을 어떻게 embedding하는지 확인한다.

- `anchor`: 기준 window
- `positive`: 같은 춤에서 가까운 시점의 window라 cosine similarity가 높게 나오는 것을 기대
- `negative`: 다른 춤의 window라 anchor와의 cosine similarity가 낮게 나오는 것을 기대

시각화는 raw MediaPipe 좌표의 12개 dance joint를 사용하고, 유사도 계산은 서비스 입력과 같은 전처리된 `(T, 12, C)` window를 사용한다.


In [5]:
# Fine-tuned TFLite encoder similarity + skeleton animation.
# This cell is intentionally self-contained so it still works after a kernel restart.
from pathlib import Path
import json
import sys

import numpy as np
import matplotlib.pyplot as plt
from matplotlib.animation import FuncAnimation
from IPython.display import HTML
import tensorflow as tf


def find_project_root_for_visualization(start=None):
    start = Path(start or Path.cwd()).resolve()
    candidates = [start, *start.parents]
    for base in [start, *start.parents]:
        candidates.extend([base / 'boot_env' / 'pjt_main', base / 'pjt_main'])
    for candidate in candidates:
        if (candidate / 'scripts' / 'finetune_embedding.py').exists():
            return candidate
    raise RuntimeError('pjt_main project root를 찾지 못했습니다.')


PROJECT_ROOT = globals().get('PROJECT_ROOT') or find_project_root_for_visualization()
PROJECT_ROOT = Path(PROJECT_ROOT).resolve()
for import_path in (PROJECT_ROOT, PROJECT_ROOT / 'src'):
    import_path = str(import_path)
    if import_path not in sys.path:
        sys.path.insert(0, import_path)

from pose.landmark_utils import DANCE_JOINTS
from scoring.scratch_features import build_pose_window

OUTPUT_DIR = Path(globals().get('OUTPUT_DIR', PROJECT_ROOT / 'data' / 'models' / 'embedding'))
reference_dir = PROJECT_ROOT / 'data' / 'reference_dances'

# If you want a specific model, set VIS_MODEL_NAME manually before running this cell.
def choose_visual_model():
    preferred = []
    if globals().get('VIS_MODEL_NAME'):
        preferred.append(str(globals()['VIS_MODEL_NAME']))
    if globals().get('BEST_MODEL_NAME'):
        preferred.append(str(globals()['BEST_MODEL_NAME']))
    if globals().get('FINETUNE_RESULTS'):
        preferred.extend(str(result['name']) for result in FINETUNE_RESULTS if result.get('name'))

    existing_meta = sorted(OUTPUT_DIR.glob('*_meta.json'), key=lambda p: p.stat().st_mtime, reverse=True)
    existing_names = [p.name.replace('_meta.json', '') for p in existing_meta]
    preferred.extend(existing_names)

    for name in preferred:
        if (OUTPUT_DIR / f'{name}.tflite').exists() and (OUTPUT_DIR / f'{name}_meta.json').exists():
            return name
    raise FileNotFoundError(f'No fine-tuned embedding TFLite/meta pair found in {OUTPUT_DIR}')


VIS_MODEL_NAME = 'embedding_gcn_e64_triplet' #choose_visual_model()
model_path = OUTPUT_DIR / f'{VIS_MODEL_NAME}.tflite'
meta_path = OUTPUT_DIR / f'{VIS_MODEL_NAME}_meta.json'
meta = json.loads(meta_path.read_text(encoding='utf-8'))
config = meta.get('encoder_config', {}) or meta.get('config', {}) or {}
input_shape = meta.get('input_shape') or [1, 30, 12, 2]
SEQUENCE_LENGTH_VIS = int(config.get('sequence_length', input_shape[1]))
FEATURE_DIMS_VIS = int(config.get('feature_dims', input_shape[-1]))

# (label, dance_name, end_frame). Missing dance names are replaced by available references.
requested_samples = [
    ('anchor',   'cheerup_dance', 300),
    ('positive', 'cheerup_dance', 308),
    ('negative', 'hiphop_move',   300),
]
available_dances = sorted(p.parent.name for p in reference_dir.glob('*/reference.npy'))
assert available_dances, f'No reference.npy files found in {reference_dir}'

def existing_dance(name, fallback_index=0):
    if name in available_dances:
        return name
    return available_dances[min(fallback_index, len(available_dances) - 1)]

SAMPLES = [
    (requested_samples[0][0], existing_dance(requested_samples[0][1], 0), requested_samples[0][2]),
    (requested_samples[1][0], existing_dance(requested_samples[1][1], 0), requested_samples[1][2]),
    (requested_samples[2][0], existing_dance(requested_samples[2][1], 1), requested_samples[2][2]),
]

# 12 dance joints local index edges.
DANCE_EDGES = [
    (0, 1), (6, 7),
    (0, 6), (1, 7),
    (0, 2), (2, 4),
    (1, 3), (3, 5),
    (6, 8), (8, 10),
    (7, 9), (9, 11),
]

COLOR_BY_LABEL = {
    'anchor': 'tab:gray',
    'reference': 'tab:gray',
    'positive': 'tab:green',
    'negative': 'tab:red',
}
DEFAULT_COLOR = 'tab:blue'

rng = np.random.default_rng(0)
entries = []
for label, dance_name, end_frame in SAMPLES:
    seq_path = reference_dir / dance_name / 'reference.npy'
    seq = np.load(seq_path, allow_pickle=True).astype(np.float32)
    if len(seq) < SEQUENCE_LENGTH_VIS:
        raise ValueError(f'{dance_name} is too short: {len(seq)} frames < {SEQUENCE_LENGTH_VIS}')
    if end_frame is None:
        end_frame = int(rng.integers(SEQUENCE_LENGTH_VIS - 1, len(seq)))
    end_frame = max(SEQUENCE_LENGTH_VIS - 1, min(int(end_frame), len(seq) - 1))
    start_frame = end_frame - SEQUENCE_LENGTH_VIS + 1

    window = build_pose_window(
        seq,
        end_frame,
        SEQUENCE_LENGTH_VIS,
        target_joints=DANCE_JOINTS,
        feature_dims=FEATURE_DIMS_VIS,
    )
    raw_window = seq[start_frame:end_frame + 1][:, DANCE_JOINTS, :2].copy()
    raw_window = np.nan_to_num(raw_window, nan=0.0, posinf=0.0, neginf=0.0)
    entries.append({
        'label': label,
        'dance': dance_name,
        'end': end_frame,
        'window': window,
        'raw_window': raw_window,
    })

interp = tf.lite.Interpreter(model_path=str(model_path))
interp.allocate_tensors()
in_det = interp.get_input_details()[0]
out_det = interp.get_output_details()[0]


def quantize_if_needed(tensor, detail):
    dtype = detail.get('dtype', np.float32)
    if np.issubdtype(dtype, np.floating):
        return tensor.astype(dtype)
    scale, zero = detail.get('quantization', (0.0, 0))
    if not scale:
        return tensor.astype(dtype)
    q = np.round(tensor / scale + zero)
    info = np.iinfo(dtype)
    return np.clip(q, info.min, info.max).astype(dtype)


def dequantize_if_needed(out, detail):
    dtype = detail.get('dtype', out.dtype)
    if np.issubdtype(dtype, np.floating):
        return out.astype(np.float32)
    scale, zero = detail.get('quantization', (0.0, 0))
    if scale:
        return (out.astype(np.float32) - zero) * scale
    return out.astype(np.float32)


def embed(window):
    tensor = window[None, ...].astype(np.float32)
    expected = tuple(int(v) for v in in_det['shape'][1:])
    if tuple(tensor.shape[1:]) != expected:
        raise ValueError(f'Model expects input {expected}, got {tuple(tensor.shape[1:])}')
    interp.set_tensor(in_det['index'], quantize_if_needed(tensor, in_det))
    interp.invoke()
    out = interp.get_tensor(out_det['index'])
    emb = dequantize_if_needed(out, out_det).reshape(-1)
    norm = np.linalg.norm(emb)
    return emb / norm if norm > 1e-8 else emb


def cosine(a, b):
    a = np.nan_to_num(np.asarray(a, dtype=np.float32))
    b = np.nan_to_num(np.asarray(b, dtype=np.float32))
    na, nb = np.linalg.norm(a), np.linalg.norm(b)
    if na < 1e-8 or nb < 1e-8:
        return 0.0
    return float(np.dot(a, b) / (na * nb))


for entry in entries:
    entry['emb'] = embed(entry['window'])

labels = [entry['label'] for entry in entries]
print(f'[MODEL] {VIS_MODEL_NAME}')
print('PROJECT_ROOT =', PROJECT_ROOT)
print('input_shape =', in_det['shape'], 'output_shape =', out_det['shape'])
print('sequence_length =', SEQUENCE_LENGTH_VIS, 'feature_dims =', FEATURE_DIMS_VIS)
print('training_summary =', meta.get('training_summary', {}))
print()
print('samples:')
for entry in entries:
    print(f" - {entry['label']:>8s}: {entry['dance']} end={entry['end']}")
print()
print('pairwise cosine similarity:')
print(' ' * 12 + ''.join(f'{label:>11s}' for label in labels))
for ei in entries:
    row = f'{ei["label"]:>10s}: '
    for ej in entries:
        row += f'{cosine(ei["emb"], ej["emb"]):+10.4f} '
    print(row)


def axis_limits_raw(raw_windows, pad=0.03):
    arr = np.stack(raw_windows, axis=0)
    x, y = arr[..., 0], arr[..., 1]
    valid = np.isfinite(x) & np.isfinite(y) & (x > 0.01) & (y > 0.01)
    if not np.any(valid):
        return (0.0, 1.0), (0.0, 1.0)
    valid_x = x[valid]
    valid_y = y[valid]
    return (
        (float(valid_x.min()) - pad, float(valid_x.max()) + pad),
        (float(valid_y.min()) - pad, float(valid_y.max()) + pad),
    )


def draw_skeleton_raw(ax, pts, color):
    pts = np.nan_to_num(pts, nan=0.0, posinf=0.0, neginf=0.0)
    xs, ys = pts[:, 0], -pts[:, 1]
    scatter = ax.scatter(xs, ys, c=color, s=30, zorder=3)
    lines = []
    for a, b in DANCE_EDGES:
        line = ax.plot([xs[a], xs[b]], [ys[a], ys[b]], color=color, lw=1.8, alpha=0.85)[0]
        lines.append(line)
    return scatter, lines


xlim, (y_lo, y_hi) = axis_limits_raw([entry['raw_window'] for entry in entries])
ylim = (-y_hi, -y_lo)
anchor_emb = entries[0]['emb']

fig, axes = plt.subplots(1, len(entries), figsize=(4 * len(entries), 4.5))
if len(entries) == 1:
    axes = [axes]

artists_by_ax = []
for ax, entry in zip(axes, entries):
    sim = cosine(anchor_emb, entry['emb'])
    ax.set_title(
        f"{entry['label']}\n{entry['dance']} (end={entry['end']})\n"
        f"cos(anchor, .) = {sim:+.3f}",
        fontsize=10,
    )
    ax.set_xlim(*xlim)
    ax.set_ylim(*ylim)
    ax.set_aspect('equal')
    ax.set_xticks([])
    ax.set_yticks([])
    artists_by_ax.append([])

frame_text = fig.text(0.5, 0.02, '', ha='center')


def update(frame_idx):
    for old in artists_by_ax:
        for artist in old:
            artist.remove()
        old.clear()
    for ax, entry, old in zip(axes, entries, artists_by_ax):
        color = COLOR_BY_LABEL.get(entry['label'], DEFAULT_COLOR)
        scatter, lines = draw_skeleton_raw(ax, entry['raw_window'][frame_idx], color)
        old.extend([scatter, *lines])
    frame_text.set_text(f'frame {frame_idx + 1} / {SEQUENCE_LENGTH_VIS}')
    artists = [frame_text]
    for old in artists_by_ax:
        artists.extend(old)
    return artists


anim = FuncAnimation(fig, update, frames=SEQUENCE_LENGTH_VIS, interval=1000 / 30, blit=False)
plt.close(fig)
HTML(anim.to_jshtml())


[MODEL] embedding_gcn_e64_triplet
PROJECT_ROOT = /workspace/users/yijin/boot_env/pjt_main
input_shape = [ 1 30 12  2] output_shape = [ 1 64]
sequence_length = 30 feature_dims = 2
training_summary = {'monitor': 'val_loss', 'restore_best_weights': True, 'epochs_ran': 21, 'best_epoch': 16, 'best_val_loss': 0.0020462842658162117, 'best_epoch_loss': 0.005147154908627272}

samples:
 -   anchor: cheerup_dance end=300
 - positive: cheerup_dance end=308
 - negative: hiphop_move end=300

pairwise cosine similarity:
                 anchor   positive   negative
    anchor:    +1.0000    +0.9984    +0.2834 
  positive:    +0.9984    +1.0000    +0.2944 
  negative:    +0.2834    +0.2944    +1.0000 


## 7. Runtime Smoke Test

마지막으로 서비스와 같은 `ScratchPoseSimilarity.compute()` 경로로 fine-tuned embedding TFLite 모델을 호출한다. 이 테스트가 finite score를 내면 `src/main.py -s embedding`에서도 같은 방식으로 동작한다.


In [4]:
from scoring.scratch_similarity import ScratchPoseSimilarity


def embedding_runtime_smoke_test(model_name):
    model_path = OUTPUT_DIR / f'{model_name}.tflite'
    meta_path = OUTPUT_DIR / f'{model_name}_meta.json'
    assert model_path.exists(), f'Missing model: {model_path}'
    meta = json.loads(meta_path.read_text(encoding='utf-8')) if meta_path.exists() else {}
    cfg = meta.get('encoder_config', {}) or meta.get('config', {})
    seq_len = int(cfg.get('sequence_length', 30))
    feature_dims = int(cfg.get('feature_dims', 2))

    ref = np.load(DANCE_DATA_DIR / 'beginner_wave' / 'reference.npy', allow_pickle=True)
    other = np.load(DANCE_DATA_DIR / 'hiphop_move' / 'reference.npy', allow_pickle=True)
    comparator = ScratchPoseSimilarity(
        str(model_path),
        sequence_length=seq_len,
        feature_dims=feature_dims,
        input_layout='BTJC',
    )

    same = None
    for idx in range(seq_len):
        same = comparator.compute(ref[idx], ref, idx)

    comparator.reset()
    cross = None
    for idx in range(seq_len):
        cross = comparator.compute(other[idx], ref, idx)

    print(f'{model_name:36s} same={same} cross={cross} finite={np.isfinite(same) and np.isfinite(cross)}')


models_to_check = [result['name'] for result in FINETUNE_RESULTS] if 'FINETUNE_RESULTS' in globals() else []
if not models_to_check:
    models_to_check = [p.name.replace('.tflite', '') for p in sorted(OUTPUT_DIR.glob('*.tflite'))]

for model_name in models_to_check:
    embedding_runtime_smoke_test(model_name)


embedding_gcn_e32_triplet            same=1.0 cross=0.41621801257133484 finite=True
embedding_gcn_e64_triplet            same=0.9999999403953552 cross=0.0 finite=True
embedding_tcn_e32_triplet            same=0.9999998807907104 cross=0.7835528254508972 finite=True
embedding_tcn_e64_triplet            same=0.9999998807907104 cross=0.2746788561344147 finite=True


## 8. 서비스 실행 명령

fine-tuned `.tflite` 모델은 `embedding` score method에서 사용한다. 내부적으로는 scratch runtime과 같은 window → embedding → cosine 계약을 쓰지만, 모델은 MPOSE2021 pretraining 후 댄스 reference로 fine-tuning된 encoder다.


In [ ]:
# Pick one model and show the service command.
BEST_MODEL_NAME = FINETUNE_RESULTS[0]['name'] if FINETUNE_RESULTS else None
if BEST_MODEL_NAME:
    print('Run service with:')
    print(f'python src/main.py -s embedding --embedding-model-name {BEST_MODEL_NAME}')
    print()
    print('Or with direct path:')
    print(f'python src/main.py -s embedding --embedding-model-path data/models/embedding/{BEST_MODEL_NAME}.tflite')
else:
    print('No fine-tuned models found yet.')

The fine-tuned `.tflite` files are used by the runtime `embedding` score method. Internally this uses the same window-to-embedding-to-cosine contract as scratch scoring, but the model starts from MPOSE2021 pretraining and is then adapted to the local dance references.